In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [2]:

IMG_SIZE = (224,224)
BATCH_SIZE = 32
DATASET_DIR = 'leaf_dataset_small'

In [3]:
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_generator = datagen.flow_from_directory(
    DATASET_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training'
)

val_generator = datagen.flow_from_directory(
    DATASET_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation'
)

Found 1600 images belonging to 2 classes.
Found 400 images belonging to 2 classes.


In [4]:
base_model = MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

In [5]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        81,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,340,033 (8.93 MB)

 Trainable params: 82,049 (320.50 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [6]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10
)

model.save("model/leaf_detector.keras")

Epoch 1/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 25s 444ms/step - accuracy: 0.9400 - loss: 0.1517 - val_accuracy: 0.9475 - val_loss: 0.1947
Epoch 2/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 21s 417ms/step - accuracy: 0.9887 - loss: 0.0342 - val_accuracy: 0.9475 - val_loss: 0.2179
Epoch 3/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 21s 411ms/step - accuracy: 0.9962 - loss: 0.0143 - val_accuracy: 0.9475 - val_loss: 0.2525
Epoch 4/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 21s 418ms/step - accuracy: 0.9981 - loss: 0.0093 - val_accuracy: 0.9525 - val_loss: 0.2248
Epoch 5/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 21s 418ms/step - accuracy: 0.9994 - loss: 0.0060 - val_accuracy: 0.9525 - val_loss: 0.2322
Epoch 6/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 21s 414ms/step - accuracy: 1.0000 - loss: 0.0032 - val_accuracy: 0.9500 - val_loss: 0.2562
Epoch 7/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 21s 413ms/step - accuracy: 1.0000 - loss: 0.0033 - val_accuracy: 0.9500 - val_loss: 0.3028
Epoch 8/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 21s 416ms/step - accuracy: 1.0000 - loss: 0.0019 - val_accu

In [10]:
import numpy as np
from PIL import Image
img = Image.open("test_image.jfif")
img = img.resize((224,224))

img_array = np.array(img)/255.0
img_array = np.expand_dims(img_array, axis=0)

pred = model.predict(img_array)[0][0]

label = "not_leaf" if pred > 0.5 else "leaf"

print("Prediction:", label)
print("Confidence:", float(pred))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
Prediction: leaf
Confidence: 0.0019501687493175268


In [11]:
print(train_generator.class_indices)

{'leaf': 0, 'not_leaf': 1}


In [12]:
model.export("model/leaf_model_serving")

INFO:tensorflow:Assets written to: model/leaf_model_serving\assets


INFO:tensorflow:Assets written to: model/leaf_model_serving\assets


Saved artifact at 'model/leaf_model_serving'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_154')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  2270108538960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2270108539344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2270108539152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2270108539728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2270108538384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2270108539920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2270108537616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2270108540688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2270108540880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2270108538576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2270108540112: TensorS

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

with open("cipherplant.tflite", "wb") as f:
    f.write(tflite_model)

ValueError: File format not supported: filepath=model/leaf_model_serving. Keras 3 only supports V3 `.keras` files and legacy H5 format files (`.h5` extension). Note that the legacy SavedModel format is not supported by `load_model()` in Keras 3. In order to reload a TensorFlow SavedModel as an inference-only layer in Keras 3, use `keras.layers.TFSMLayer(model/leaf_model_serving, call_endpoint='serving_default')` (note that your `call_endpoint` might have a different name).